In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np
import scipy as sci
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc
import scirpy as ir
import anndata as ann
import awkward as ak
import muon as mu

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

# Set scanpy settings
sc.settings.verbosity = 3  # verbosity level
sc.settings.set_figure_params(dpi=80, facecolor='white')


e:\Anaconda\envs\mvTCR\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Anaconda\envs\mvTCR\lib\site-packages\airr\schema.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


In [2]:
# Define file paths
reference_path = r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data\Lee_EAE\mvTCR_processed\p1_preprocessed_with_metadata.h5ad"
query_path = r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data\EAE\GSE188320\10x_processed_all_genes.h5ad"


In [3]:
# Load reference data
print("Loading reference data...")
adata_ref = sc.read_h5ad(reference_path)
adata_ref

Loading reference data...


e:\Anaconda\envs\mvTCR\lib\site-packages\anndata\utils.py:334: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)


AnnData object with n_obs × n_vars = 51500 × 2500
    obs: 'mouse_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len', 'has_binding', 'set', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'cloned', 'in_two_tissue', 'modified_cell_type', 'state'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'aa_to_id', 'chain_indices', 'clone_id', 'clonotype', 'hvg', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors', 'mouse_id_enc'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices', 'mouse_id_ohe'

In [4]:
# adata_ref.obsm['airr'].shape
# Rename 'SP' to 'SPL' and 'CN' to 'CNS' in adata_ref.obs['tissue']
adata_ref.obs['tissue'] = adata_ref.obs['tissue'].replace({'SP': 'SPL', 'CN': 'CNS'})

airr = adata_ref.obsm["airr"]

type(airr)

awkward.highlevel.Array

In [5]:
adata_query = sc.read_h5ad(query_path)
adata_query

AnnData object with n_obs × n_vars = 28652 × 14358
    obs: 'batch', 'condition', 'mouse_id', 'assignment_demuxem', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len'
    var: 'n_cells'
    uns: 'aa_to_id', 'chain_indices', 'clone_id', 'clonotype', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices'

In [6]:
# only keep EAE sets
adata_query = adata_query[adata_query.obs['condition'] == 'EAE']
adata_query

View of AnnData object with n_obs × n_vars = 15025 × 14358
    obs: 'batch', 'condition', 'mouse_id', 'assignment_demuxem', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len'
    var: 'n_cells'
    uns: 'aa_to_id', 'chain_indices', 'clone_id', 'clonotype', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices'

In [7]:
# Rename 'assignment_demuxem' to 'tissue' in query_merged.obs
adata_query.obs = adata_query.obs.rename(columns={'assignment_demuxem': 'tissue'})
adata_query.obs['mouse_id'] = adata_query.obs.index.astype(str).str.split('-').str[-1]

adata_query.obs['mouse_id']

AAACCTGAGGCTCAGA-1-b6m1    b6m1
AAACCTGCAATAGCAA-1-b6m1    b6m1
AAACCTGCAGTCTTCC-1-b6m1    b6m1
AAACCTGCATTCTTAC-1-b6m1    b6m1
AAACCTGGTACATCCA-1-b6m1    b6m1
                           ... 
TTTGTCATCAACACTG-1-b7m2    b7m2
TTTGTCATCACGAAGG-1-b7m2    b7m2
TTTGTCATCCGTAGGC-1-b7m2    b7m2
TTTGTCATCCTTAATC-1-b7m2    b7m2
TTTGTCATCGTTGACA-1-b7m2    b7m2
Name: mouse_id, Length: 15025, dtype: object

In [8]:
adata_query.obs = adata_query.obs[['tissue', 'mouse_id']].copy()
adata_ref.obs = adata_ref.obs[['tissue', 'mouse_id']].copy()


In [9]:
import anndata as ad
def merge_anndata_to_base(anndata1, anndata2):
    c_gene = anndata2.var_names.intersection(anndata1.var_names)

    missing_genes = anndata1.var_names.difference(anndata2.var_names)

    # Pad missing genes with zeros (assumes dense arrays, can be adapted for sparse)
    if len(missing_genes) > 0:
        import pandas as pd
        import scipy.sparse

        shape = (anndata2.n_obs, len(missing_genes))
        X_pad = np.zeros(shape, dtype=anndata2.X.dtype)
        X_pad = scipy.sparse.csr_matrix(X_pad) if isinstance(anndata2.X, scipy.sparse.spmatrix) else X_pad

        # Create dummy .var for missing genes
        var_pad = pd.DataFrame(index=missing_genes)

        # Create a temporary AnnData with padded genes
        adata_pad = ad.AnnData(X=X_pad, obs=anndata2.obs.copy(), var=var_pad)

        # Add missing genes and reorder to match anndata1
        anndata2_full = ad.concat([anndata2, adata_pad], axis=1, join="outer", merge="first" )
        anndata2_full = anndata2_full[:, anndata1.var_names]

    else:
        anndata2_full = anndata2[:, anndata1.var_names]
    return anndata2_full


In [10]:
# Add dataset labels to obs
query_merged = merge_anndata_to_base(adata_ref, adata_query)

assert set(query_merged.var_names) == set(adata_ref.var_names), "Gene alignment failed!"
print("✓ Gene alignment successful!")

query_merged


✓ Gene alignment successful!


View of AnnData object with n_obs × n_vars = 15025 × 2500
    obs: 'tissue', 'mouse_id'
    var: 'n_cells'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices'

In [11]:
ad_merged = ad.concat([adata_ref, query_merged], axis=0, join="outer", merge='first')
ad_merged

e:\Anaconda\envs\mvTCR\lib\site-packages\anndata\utils.py:334: ExperimentalFeatureWarning: Outer joins on awkward.Arrays will have different return values in the future. For details, and to offer input, please see:

	https://github.com/scverse/anndata/issues/898
  warnings.warn(msg, category, stacklevel=stacklevel)


AnnData object with n_obs × n_vars = 66525 × 2500
    obs: 'tissue', 'mouse_id'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices', 'mouse_id_ohe'

In [12]:
# two awkward arrays with the same fields (schema)
airr_merged = ak.concatenate([adata_ref.obsm['airr'], adata_query.obsm['airr']], axis=0)
ad_merged.obsm['airr'] = airr_merged

# Remove all keys from obsm except 'airr'
keys_to_remove = [k for k in ad_merged.obsm.keys() if k != 'airr']
for k in keys_to_remove:
    del ad_merged.obsm[k]
ad_merged

AnnData object with n_obs × n_vars = 66525 × 2500
    obs: 'tissue', 'mouse_id'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    obsm: 'airr'

In [13]:
print(ad_merged.obs['tissue'])
print(ad_merged.obs['mouse_id'])

AAACCAAAGGGGAGCT-1_0516_CNS    CNS
AAACCAGCACGTAAAG-1_0516_CNS    CNS
AAACCATTCCTCCGGT-1_0516_CNS    CNS
AAACCCATCAGTATCG-1_0516_CNS    CNS
AAACCCCAGCCTAAGC-1_0516_CNS    CNS
                              ... 
TTTGTCATCAACACTG-1-b7m2        SPL
TTTGTCATCACGAAGG-1-b7m2        CNS
TTTGTCATCCGTAGGC-1-b7m2        NaN
TTTGTCATCCTTAATC-1-b7m2        CNS
TTTGTCATCGTTGACA-1-b7m2        SPL
Name: tissue, Length: 66525, dtype: category
Categories (7, object): ['CNS', 'COL', 'DLN', 'MLN', 'PP', 'SI', 'SPL']
AAACCAAAGGGGAGCT-1_0516_CNS     5_3
AAACCAGCACGTAAAG-1_0516_CNS     5_3
AAACCATTCCTCCGGT-1_0516_CNS     5_4
AAACCCATCAGTATCG-1_0516_CNS     5_3
AAACCCCAGCCTAAGC-1_0516_CNS     5_3
                               ... 
TTTGTCATCAACACTG-1-b7m2        b7m2
TTTGTCATCACGAAGG-1-b7m2        b7m2
TTTGTCATCCGTAGGC-1-b7m2        b7m2
TTTGTCATCCTTAATC-1-b7m2        b7m2
TTTGTCATCGTTGACA-1-b7m2        b7m2
Name: mouse_id, Length: 66525, dtype: object


# TCR process again

In [14]:
ir.pp.index_chains(ad_merged)
ir.tl.chain_qc(ad_merged)

# by nucleotide acid seq
# ir.pp.ir_dist(ad_merged)
# ir.tl.define_clonotypes(ad_merged, receptor_arms="all", dual_ir="primary_only")

  0%|          | 0/14 [00:00<?, ?it/s]

100%|██████████| 14/14 [00:16<00:00,  1.16s/it]


Stored result in `adata.obs["receptor_type"]`.
Stored result in `adata.obs["receptor_subtype"]`.
Stored result in `adata.obs["chain_pairing"]`.


In [15]:
junction_aa = ir.get.airr(ad_merged, "junction_aa")
cells_with_both_chains = ~(junction_aa['VJ_1_junction_aa'].isna() | junction_aa['VDJ_1_junction_aa'].isna())
print(ad_merged.shape)
ad_merged = ad_merged[cells_with_both_chains]
print(ad_merged.shape)

(66525, 2500)
(66525, 2500)


In [16]:
from mvtcr.utils_preprocessing import Preprocessing

In [17]:
Preprocessing.encode_clonotypes(ad_merged)

 80%|████████  | 40000/49801 [01:49<00:26, 365.45it/s]


MemoryError: 

In [ ]:
Preprocessing.encode_tcr(ad_merged, airr_name='junction_aa', 
               alpha_label_key='alpha_seq', alpha_length_key='alpha_len',
               beta_label_key='beta_seq', beta_length_key='beta_len')

In [ ]:
ad_merged.write('merged_EAE.h5ad')